In [ ]:
# ============================================================
# GENERIC PANDAS DATA CLEANING PIPELINE
# ============================================================
#
# OBJECTIF
# --------
# Construire un pipeline générique capable de :
#
# 1. Explorer un dataset inconnu
# 2. Détecter automatiquement les types probables
# 3. Nettoyer les données
# 4. Convertir les colonnes vers les bons types
# 5. Vérifier le résultat final
#
# Utilisable sur n'importe quel CSV ou Excel.
#
# ============================================================

import pandas as pd


# ============================================================
# 1. INSPECTION DU DATAFRAME
# ============================================================

def inspect_dataframe(df):
    """
    Résumé rapide du dataset
    """

    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)

    print(f"\nNombre de lignes : {df.shape[0]}")
    print(f"Nombre de colonnes : {df.shape[1]}")

    print("\nTypes détectés :")
    print(df.dtypes)

    print("\nValeurs manquantes :")
    print(df.isnull().sum())

    print("\nMémoire utilisée :")
    print(
        round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        ),
        "MB"
    )

    print("\nAperçu :")
    print(df.head())


# ============================================================
# 2. FONCTIONS DE CONVERSION
# ============================================================

def to_int(series):
    """
    Conversion vers entier nullable Pandas.
    Accepte les valeurs manquantes.
    """

    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .astype("Int64")
    )


def to_float(series):
    """
    Conversion vers float.
    Les erreurs deviennent NaN.
    """

    return pd.to_numeric(
        series,
        errors="coerce"
    )


def clean_text(series):
    """
    Nettoyage standard du texte.
    """

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


def clean_id(series):
    """
    Nettoyage des identifiants.
    """

    return (
        series
        .astype(str)
        .str.strip()
        .str.upper()
    )


def to_datetime_col(series):
    """
    Conversion vers datetime.
    """

    return pd.to_datetime(
        series,
        errors="coerce"
    )


# ============================================================
# 3. DÉTECTION AUTOMATIQUE
# ============================================================

def detect_numeric_columns(df, threshold=0.80):
    """
    Détecte les colonnes majoritairement numériques.

    threshold=0.80
    signifie :
    au moins 80% des valeurs convertibles
    """

    numeric_cols = []

    for col in df.columns:

        converted = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            numeric_cols.append(col)

    return numeric_cols


def detect_date_columns(df, threshold=0.80):
    """
    Détecte les colonnes majoritairement dates.
    """

    date_cols = []

    for col in df.columns:

        converted = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        success_rate = converted.notna().mean()

        if success_rate >= threshold:

            if col not in date_cols:
                date_cols.append(col)

    return date_cols


def detect_category_columns(
    df,
    max_unique_ratio=0.10
):
    """
    Détecte les colonnes candidates au type category.

    Exemple :

    1000 lignes
    5 villes

    5 / 1000 = 0.005

    => bonne candidate
    """

    category_cols = []

    object_cols = df.select_dtypes(
        include="object"
    )

    for col in object_cols:

        ratio = (
            df[col].nunique()
            /
            len(df)
        )

        if ratio <= max_unique_ratio:
            category_cols.append(col)

    return category_cols


# ============================================================
# 4. CONVERSION AUTOMATIQUE
# ============================================================

def convert_numeric_columns(
    df,
    numeric_cols
):
    """
    Convertit les colonnes numériques.
    """

    for col in numeric_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df


def convert_date_columns(
    df,
    date_cols
):
    """
    Convertit les colonnes dates.
    """

    for col in date_cols:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

    return df


def convert_category_columns(
    df,
    category_cols
):
    """
    Nettoyage + conversion category.
    """

    for col in category_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .astype("category")
        )

    return df


# ============================================================
# 5. NETTOYAGE TEXTE GÉNÉRIQUE
# ============================================================

def clean_remaining_text(df):

    object_cols = df.select_dtypes(
        include="object"
    )

    for col in object_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

    return df


# ============================================================
# 6. PIPELINE PRINCIPAL
# ============================================================

def auto_clean_dataframe(df):

    print("\n" + "=" * 60)
    print("AUTO DETECTION STARTED")
    print("=" * 60)

    # --------------------
    # Détection
    # --------------------

    numeric_cols = detect_numeric_columns(df)

    date_cols = detect_date_columns(df)

    category_cols = detect_category_columns(df)

    print("\nColonnes numériques détectées :")
    print(numeric_cols)

    print("\nColonnes dates détectées :")
    print(date_cols)

    print("\nColonnes catégorielles détectées :")
    print(category_cols)

    # --------------------
    # Conversions
    # --------------------

    df = convert_numeric_columns(
        df,
        numeric_cols
    )

    df = convert_date_columns(
        df,
        date_cols
    )

    df = convert_category_columns(
        df,
        category_cols
    )

    df = clean_remaining_text(df)

    print("\nNettoyage terminé.")

    return df


# ============================================================
# 7. UTILISATION
# ============================================================

# Charger le dataset

df = pd.read_csv("data.csv")

# Inspection avant nettoyage

inspect_dataframe(df)

# Nettoyage automatique

df = auto_clean_dataframe(df)

# Vérification finale

print("\n" + "=" * 60)
print("AFTER CLEANING")
print("=" * 60)

print(df.dtypes)

print("\nValeurs manquantes :")
print(df.isnull().sum())

print("\nMémoire utilisée :")
print(
    round(
        df.memory_usage(deep=True).sum()
        / 1024**2,
        2
    ),
    "MB"
)

print("\nAperçu final :")
print(df.head())


# ============================================================
# COMMENT PENSER ?
# ============================================================
#
# Pour chaque colonne :
#
# Est-ce un nombre ?
#      ↓
# Numeric
#
# Est-ce une date ?
#      ↓
# Datetime
#
# Est-ce une catégorie répétée ?
#      ↓
# Category
#
# Est-ce un identifiant ?
#      ↓
# String
#
# PIPELINE MENTAL :
#
# read_csv()
#      ↓
# inspect_dataframe()
#      ↓
# detect_types()
#      ↓
# convert_types()
#      ↓
# verify_results()
#      ↓
# analysis()
#
# ============================================================